In [1]:
import numpy as np
import rebound
import matplotlib.pyplot as plt
import os
import pandas as pd
from datetime import datetime

os.chdir('../src/')

from prop_elem import *
from sbdynt import *

# Computing Proper Elements Using Custom Inputs

While many users will be interested in primarily just running the default TNO and asteroid analysis suite provided by the ``run_tno`` and ``run_asteroid`` functions, many users will be interested in computing the proper elements without running the machine learning and chaos indicators. 

In addition, users may want to specify other parameters, such as the length of integration, output cadence, or the planets included in the simulation.
SBDynT allows for custom runs to produce proper elements easily using the underying functions which are called directly by the ``run_tno`` and ``run_asteroid`` functions, which we will describe here. 

## Initializing Integration

To compute synthetic proper elements, we need to produce an initial REBOUND simulation of the Solar System accompanied by the particles of interest. To initialize the default simulations for TNO's and asteroid, SBDynT provides the ``setup_default_tno_integration`` and ``setup_default_asteroid_integration`` functions. 

This function does not require the small_body class meaning a user may use these functions to initialize objects outisde of the small_body class ecosystem if they wish. 

``initialize_simulation(planets=['all'], des=None, clones=0, cloning_method='Gaussian', datadir='', saveic=False, logfile=False, save_sbdb=False)``
``returns flag, epoch, sim, weights``

The primary inputs required by the user to include will be the ``planets`` and ``des``variables. The ``clones` and ``cloning_methods`` variables are primarily of interest for users wishing the run the machine_learning analysis or some chaos indicator analysis, while the rest of the variables are primarily bookkeeping parameters which are relevant to every analysis.


Below, we initialize simulations for the asteroid ``(1) Ceres`` and the TNO ``(15760) Albion``. using the recommended planets for each type of object. 

#####

In [2]:
flag1, epoch1, sim1 = initialize_simulation(planets=['inner'], des='Ceres', clones=0,  datadir='../example-notebooks/advanced_example_sims', saveic=True, logfile=True, save_sbdb=True)
flag2, epoch2, sim2 = initialize_simulation(planets=['outer'], des='Albion', clones=0, datadir='../example-notebooks/advanced_example_sims', saveic=True, logfile=True, save_sbdb=True)

We can see that in the example_sims directory, there are now several files that have been populated, including a log file for the current date, a pkl file outling the exact query made to JPL Horizons, and initial condition REBOUND simulations which are equivalent to the sim1 and sim2 simulations we have saved as variables above. 
 

## Running the Integration

Now that our simulations are initialized, we can run the integration using the ``run_simulation`` function.

```rflag, sim = integrate_for_pe(sim, des=None, archivefile=None,datadir=None, logfile=None,tmax=10e6,tout=500., direction='bf')```
                                       

We show examples integrating both Ceres and Eris, setting tmax for much shorter timescales, but sufficient to capture the slowest plamnetary secular frequencies.

In [5]:
rflag1, sim1 = integrate_for_pe(sim1, des='Ceres', datadir='../example-notebooks/advanced_example_sims', logfile=True,tmax=1e6,tout=500.)
rflag2, sim2 = integrate_for_pe(sim2, des='Albion', datadir='../example-notebooks/advanced_example_sims', logfile=True,tmax=15e6,tout=5000.)


## Reading the Simulation

Users familiar with Rebound Simulationarchives may be able to read in the orbital elements from the completed simulations themsevles, but SBDynT provides a useful function for reading in the orbital elements for the planets, the small body, and any included clones, while also sorting the orbital elements by time, and handling complex cases, such as multi-resolution simulations, like those used in the machine learning analyses.

```read_archive_for_pe(des, clones=3, datadir='',archivefile='', logfile='', object_type= None)```

In [2]:
pr_flag1, time1, sb_elems1, planet_elems1, clone_elems1, small_planets_flag1 =  read_archive_for_pe('Ceres', clones=0, datadir='../example-notebooks/advanced_example_sims', logfile=True, object_type= None)
pr_flag2, time2, sb_elems2, planet_elems2, clone_elems2, small_planets_flag2 =  read_archive_for_pe('Albion', clones=0, datadir='../example-notebooks/advanced_example_sims', logfile=True, object_type= None)

# Computing the Synthetic Proper Elements

To compute synthetic proper elements for the small_body the typical user will simply call the ``compute_proper`` function, as shown below.
The function call will print the comptued proper elements out; additional outputs, such as the uncertianties, are saved to the particle and can be retrieved directly. 

In [3]:
pflag1, pe1 = calc_proper_elements(des='Ceres', times = time1, sb_elems = sb_elems1, planet_elems = planet_elems1, small_planets_flag = small_planets_flag1)
pflag2, pe2 = calc_proper_elements(des='Albion', times = time2, sb_elems = sb_elems2, planet_elems = planet_elems2, small_planets_flag = small_planets_flag2)

### Ceres Outputs

In [5]:
print('Secular Planetary Frequencies: ',pe1.planet_freqs)

print('\nProper Elements: ', pe1.proper_elements)
print('\nProper Errors: ', pe1.proper_errors)

print('\nMean Elements: ', pe1.mean_elements)
print('\nOsculating Elements: ', pe1.osculating_elements)

Secular Planetary Frequencies:  {'g5': 2.998502019037244e-06, 'g6': 2.1989695173600555e-05, 'g7': 3.997859913795975e-06, 'g8': 9.995006730124147e-07, 's6': -2.0056755210933028e-05, 's7': -3.0248561210410637e-06, 's8': 2.9985020190372436e-06, 'g2': 4.997502306105295e-06, 'g3': 5.996992740000256e-06, 'g4': 1.3984711937330592e-05, 's3': -1.3998578069400697e-05, 's4': -1.2993515982199984e-05, 's2': -4.998472740559624e-06}

Proper Elements:  {'a': 2.763414955884124, 'e': 0.11440635880030021, 'sinI': 0.16751029453790026, 'g(rev/yr)': 4.197870671935434e-05, 's(rev/yr)': -4.594773312012324e-05, 'g("/yr)': 54.40440390828323, 's("/yr)': -59.54826212367973}

Proper Errors:  {'RMS_a': 6.926839692781587e-06, 'RMS_e': 0.0020548280111898592, 'RMS_sinI': 7.468645341562792e-05, 'RMS_g(rev/yr)': 2.1071665057585344e-08, 'RMS_s(rev/yr)': 4.251924585000367e-08, 'RMS_g("/yr)': 0.027308877914630606, 'RMS_s("/yr)': 0.055104942621604765}

Mean Elements:  {'a': 2.7634149558841243, 'e': 0.11798620410157663, 'sin

### Albion Outputs

In [7]:
print('Secular Planetary Frequencies: ',pe2.planet_freqs)

print('\nProper Elements: ', pe2.proper_elements)
print('\nProper Errors: ', pe2.proper_errors)

print('\nMean Elements: ', pe2.mean_elements)
print('\nOsculating Elements: ', pe2.osculating_elements)

Secular Planetary Frequencies:  {'g5': 3.2656078319109914e-06, 'g6': 2.1792731663553793e-05, 'g7': 2.3984798487707692e-06, 'g8': 5.328582590628065e-07, 's6': -2.0326567441892202e-05, 's7': -2.328421759390913e-06, 's8': -5.33155870430235e-07}

Proper Elements:  {'a': 43.928939588757416, 'e': 0.07188968466523353, 'sinI': 0.047291872643541254, 'g(rev/yr)': 3.330793890127398e-07, 's(rev/yr)': -3.332245992630483e-07, 'g("/yr)': 0.4316708881605108, 's("/yr)': -0.4318590806449105}

Proper Errors:  {'RMS_a': 1.4773215235317757e-05, 'RMS_e': 0.0009611377208968375, 'RMS_sinI': 0.005239381655554297, 'RMS_g(rev/yr)': 6.641259516340961e-08, 'RMS_s(rev/yr)': 3.111347866501414e-08, 'RMS_g("/yr)': 0.08607072333177886, 'RMS_s("/yr)': 0.040323068349858326}

Mean Elements:  {'a': 43.92893958875742, 'e': 0.07232054088413564, 'sinI': 0.05118341358477274, 'g(rev/yr)': 3.1573518123402486e-07, 's(rev/yr)': -2.5299576151908167e-07, 'g("/yr)': 0.4091927948792962, 's("/yr)': -0.3278825069287299}

Osculating Elem

# Using Local Simulation Data to Compute Proper Elements

Because of the input schema of the ```calc_proper_elements``` function, users are able to comptue proper elements using time arrays of orbital elements produced by other integration methods as well, as long as the inputs are given correctly. This may be helpful for users with simulation data produced using other integrators, such as Mercury, SWIFT, or ASSIST. 

We show an example here of reading in local simulation data, modifying the arrays to match the input parameters for the function, and calling the ```calc_proper_elements``` function with the inputs.

## Calling ```calc_proper_elements``` Directly

If you wish to call ```calc_proper_elements``` directly, you, the user, must provide the following inputs:

(1) ```times```          (numpy.ndarray, shape = (len(simulation))): 
    
```times``` is the time array corresponding to the orbital simulation, in units of years. 
This time array should be in increasing order*, and should have the same resolution throughout the entire array**.

(2) ```sb_elems```       (numpy.ndarray, shape = (5, len(simulation))):

```sb_elems``` contains each array of the small-body secular orbital elements, [a,e,I,omega,Omega] at the times contained in the ```times``` array. 


(3) ```planet_elems```   (dict):

```planet_elems``` is a dictionary containing the orbital arrays for each planet in the simulation. 
Each item in the dictionary is a key of the planet name, with a corresponding numpy.ndarray object with the same shape as sb_elems, shape=(5, len(simulation)).
Example: planet_elems = {'jupiter': jupiter_elems, 'saturn': saturn_elems, ...}

(4) ```small_planets_flag```   (boolean):

```small_planets_flag``` determines whether the proper element computation filters out only the giant planet frequencies, or includes the rocky planets. 

True: filters out 7 planets (venus, earth, mars, jupiter, saturn, uranus, neptune)
    
False: filters out 4 planets (jupiter, saturn, uranus, neptune)
   
    
*: Users who produce integrations for SBDynT using the machine learning functionality will have a Rebound Simulationarchive with multiple resolutions contained in the single archive. Also, using the default forwards + backwards integration will produce a Simulation archive that has both negative and positive time steps. The ```read_archive_for_pe``` function automatically handles these effects in a REBOUND archive, and returns orbital elements and a time-array that have the correct cadence and direction for proper element computation. 

**: correct values for the proper elements will still be produced if given in descending order. However, the sign on the output ```g``` and ```s``` freuqencies will be flipped.

# Using Self-Defined Planetary Frequencies

While not recommended, users may also compute proper elements without providing the planetary orbital elements. The user may do so by either providing 
(1) their own ```gs_dict```, which is a dictionary of the relevant ```g_i``` and ```s_i``` planetary frequencies, or 
(2) submitting ```None```, which will use the ```g_i``` and ```s_i``` frequencies reported by Milani and Knezevic 1994 for the giant planets, and the values for the rocky planets provided by Brouwer and van Woerkom 1950.

Note that these units should be reported in units of rev/yr.

Here is an example of running ```calc_proper_elements``` with a self-defined ```gs_dict``` equal to the 1950 analytical values reported by Brouwer and van Woerkom, and with a ```gs_dict = None```, whic automatically uses the default values provided by Milani and Knezevic 1994.

In [12]:
# Brouwer and van Woerkom 1950 Frequencies for all 7 Planets
gs_dict = {'g5': 4.29591/1296000 ,'g6': 27.77406/1296000, 'g7': 2.71931/1296000, 'g8': 0.63332/1296000, 
           's6': -25.73355/1296000, 's7': -2.90266/1296000, 's8': -0.67752/1296000,
          'g2': 7.34474/1296000, 'g3': 17.32832/1296000, 'g4': 18.00233/1296000,
          's2': -6.57080/1296000, 's3': -18.74359/1296000, 's4': -17.63331/1296000}

pflag1_new, pe1_new = calc_proper_elements(des='Ceres', times = time1, sb_elems = sb_elems1, planet_elems = None, small_planets_flag = small_planets_flag1, gs_dict = gs_dict)
pflag2_new, pe2_new = calc_proper_elements(des='Albion', times = time2, sb_elems = sb_elems2, planet_elems = None, small_planets_flag = small_planets_flag2, gs_dict = None)

### Ceres Outputs

In [13]:
print('Secular Planetary Frequencies: ',pe1_new.planet_freqs)

print('\nProper Elements: ', pe1_new.proper_elements)
print('\nProper Errors: ', pe1_new.proper_errors)

print('\nMean Elements: ', pe1_new.mean_elements)
print('\nOsculating Elements: ', pe1_new.osculating_elements)

Secular Planetary Frequencies:  {'g5': 3.3147453703703705e-06, 'g6': 2.143060185185185e-05, 'g7': 2.098233024691358e-06, 'g8': 4.886728395061728e-07, 's6': -1.985613425925926e-05, 's7': -2.239706790123457e-06, 's8': -5.227777777777778e-07, 'g2': 5.6672376543209875e-06, 'g3': 1.3370617283950618e-05, 'g4': 1.3890686728395062e-05, 's2': -5.070061728395062e-06, 's3': -1.4462646604938273e-05, 's4': -1.3605949074074075e-05}

Proper Elements:  {'a': 2.763414955884123, 'e': 0.11440684394337643, 'sinI': 0.16751063800387628, 'g(rev/yr)': 4.197870671935434e-05, 's(rev/yr)': -4.594773312012324e-05, 'g("/yr)': 54.40440390828323, 's("/yr)': -59.54826212367973}

Proper Errors:  {'RMS_a': 6.926839693835333e-06, 'RMS_e': 0.0020387846983011605, 'RMS_sinI': 0.00014313761733244913, 'RMS_g(rev/yr)': 2.1071665057585344e-08, 'RMS_s(rev/yr)': 4.2730839319255134e-08, 'RMS_g("/yr)': 0.027308877914630606, 'RMS_s("/yr)': 0.05537916775775465}

Mean Elements:  {'a': 2.7634149558841243, 'e': 0.11798620410157663, 'si

### Albion Outputs

In [14]:
print('Secular Planetary Frequencies: ',pe2_new.planet_freqs)

print('\nProper Elements: ', pe2_new.proper_elements)
print('\nProper Errors: ', pe2_new.proper_errors)

print('\nMean Elements: ', pe2_new.mean_elements)
print('\nOsculating Elements: ', pe2_new.osculating_elements)

Secular Planetary Frequencies:  {'g5': 3.299e-06, 'g6': 2.197e-05, 'g7': 2.398e-06, 'g8': 5.022e-07, 's6': -2.032e-05, 's7': -2.309e-06, 's8': -5.3395e-07}

Proper Elements:  {'a': 43.928939588757416, 'e': 0.07188713191357697, 'sinI': 0.04729187721812999, 'g(rev/yr)': 3.330793890127398e-07, 's(rev/yr)': -3.332245992630483e-07, 'g("/yr)': 0.4316708881605108, 's("/yr)': -0.4318590806449105}

Proper Errors:  {'RMS_a': 1.4773215235505487e-05, 'RMS_e': 0.0009592827070726014, 'RMS_sinI': 0.005238274140608541, 'RMS_g(rev/yr)': 6.641259516340961e-08, 'RMS_s(rev/yr)': 3.111347866501414e-08, 'RMS_g("/yr)': 0.08607072333177886, 'RMS_s("/yr)': 0.040323068349858326}

Mean Elements:  {'a': 43.92893958875742, 'e': 0.07232054088413564, 'sinI': 0.05118341358477274, 'g(rev/yr)': 3.1573518123402486e-07, 's(rev/yr)': -2.5299576151908167e-07, 'g("/yr)': 0.4091927948792962, 's("/yr)': -0.3278825069287299}

Osculating Elements:  {'a': array([43.93250143]), 'e': array([0.06937744]), 'I': array([0.03814445]), 

### Note that while mostly similar, there are very minor differences in the reported proper elements and the uncertainties when using self-defined planetary secular frequencies in these cases. 

### However, the impact of self-reported freuqencies could be inflated in cases of small bodies which are really secularly resonant with a planet. Therefore, we recommend using planetary orbital element arrays when possible. 